In [14]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client('scaler-sql-476312')

# **JOINS**

In [2]:
# How do we know the primary key basically it will mention in ER Diagram if they didn't there is a way we can find the primary key
query = """
SELECT count(*) as no_of_rows , count(customer_id) as unique
FROM `farmers_market.customer_purchases`
"""
df = client.query(query).to_dataframe()
df

# Since they both are same so customer_id is a Primary key for customer table

,no_of_rows,unique
0,1003,1003


In [3]:
 # Question - – Get a list of customers' zip codes for customers who made a purchase on 2019-04-06.
query = """
SELECT cu.customer_zip
FROM `farmers_market.customer` cu
LEFT JOIN `farmers_market.customer_purchases` cp
  ON cu.customer_id = cp.customer_id
WHERE cp.market_date = "2019-04-06"
"""

df = client.query(query).to_dataframe()
df

,customer_zip
0,22801
1,22801
2,22801
3,22801
4,22801
5,22821
6,22821
7,22821


In [4]:
 # Question - – Get the list of customers who have never made any purchase
query = """
SELECT cu.customer_id,cu.customer_first_name
FROM `farmers_market.customer` cu
LEFT JOIN `farmers_market.customer_purchases` cp
  USING (customer_id)
WHERE cp.customer_id IS NULL
"""

df = client.query(query).to_dataframe()
df

,customer_id,customer_first_name
0,56,Rohit
1,55,James


In [5]:
 # Question - – There are customers who's information is deleted from the customers table who had previously made the purchase.
query = """
SELECT cp.customer_id
FROM `farmers_market.customer` cu
RIGHT JOIN `farmers_market.customer_purchases` cp
  ON cu.customer_id = cp.customer_id
WHERE cu.customer_id IS NULL
"""

df = client.query(query).to_dataframe()
df

,customer_id
0,57
1,58
2,59


In [6]:
# Question -- Find out the customers who are either new to the market or have deleted their account from the market.
query = """
SELECT cp.customer_id AS deleted_cust, cu.customer_id AS new_cust
FROM `farmers_market.customer` cu
FULL JOIN `farmers_market.customer_purchases` cp
  ON cu.customer_id = cp.customer_id
WHERE cu.customer_id IS NULL OR cp.customer_id IS NULL
"""
df = client.query(query).to_dataframe()
df

,deleted_cust,new_cust
0,<NA>,56
1,<NA>,55
2,57,<NA>
3,58,<NA>
4,59,<NA>


# **UNION DISTINCT FOR BIQ QUERY / UNION FOR SQL**

In Below code we find the who are new customers and who are the deleted customers for that we use **UNION DISTINCT for Bigquery / UNION for SQL**

# **"AS Column_name"**

In Final result we got the customers who are new and who are deleted but we don't know which customer is new or deleted so we use **AS COL_NAME**

In below question we use as and give the col_name as type so we can identify which customer is new or deleted

In [7]:
#Question -- Find out the customers who are either new to the market or have deleted their account from the market. [Same above Question but we will find the results by using UNION instead of full join]
query = """
(
  SELECT cu.customer_id , "New_Customer" AS type
  FROM `farmers_market.customer` cu
  LEFT JOIN `farmers_market.customer_purchases` cp
    ON cu.customer_id = cp.customer_id
  WHERE cp.customer_id IS NULL
)
UNION DISTINCT
(
  SELECT cp.customer_id,"Deleted_Customer" AS type
  FROM `farmers_market.customer` cu
  RIGHT JOIN `farmers_market.customer_purchases` cp
    ON cu.customer_id = cp.customer_id
  WHERE cu.customer_id IS NULL
)

"""
df = client.query(query).to_dataframe()
df

,customer_id,type
0,56,New_Customer
1,55,New_Customer
2,57,Deleted_Customer
3,58,Deleted_Customer
4,59,Deleted_Customer


# **Joining more than 2 or 3 tables**

In [10]:
# Question  -- Get details of the top 5 most rented films.
query = """
SELECT F.film_id, F.title, COUNT(R.rental_id) as total_rental
FROM `cineflix.rental` R
INNER JOIN `cineflix.inventory` I
  ON I.inventory_id = R.inventory_id
INNER JOIN `cineflix.film` F
  ON F.film_id = I.film_id
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 5
"""
df = client.query(query).to_dataframe()
df

,film_id,title,total_rental
0,103,BUCKET BROTHERHOOD,34
1,738,ROCKETEER MOTHER,33
2,767,SCALAWAG DUCK,32
3,489,JUGGLER HARDLY,32
4,730,RIDGEMONT SUBMARINE,32


In [11]:
# Question  -- Get a list of all customers who have rented more films than the average number of rentals.
query = """
SELECT C.customer_id, C.first_name, COUNT(R.rental_id) AS tot_rental
FROM `cineflix.rental` R
INNER JOIN `cineflix.customer` C
  ON C.customer_id = R.customer_id
GROUP BY 1, 2
HAVING
  tot_rental > (
    SELECT round(avg(total_rentals)) AS avg_rental
    FROM
      (
        SELECT customer_id, COUNT(*) AS total_rentals
        FROM `cineflix.rental`
        GROUP BY 1
      )
  )
"""
df = client.query(query).to_dataframe()
df

,customer_id,first_name,tot_rental
0,1,MARY,32
1,5,ELIZABETH,38
2,6,JENNIFER,28
3,7,MARIA,33
4,12,NANCY,28
...,...,...,...
250,588,MARION,29
251,589,TRACY,28
252,592,TERRANCE,29
253,595,TERRENCE,30


In [12]:
# Question    -- List all customers and the total amount they have spent on rentals, including customers who have never rented a film.
query = """
SELECT C.customer_id, round(sum(P.amount)) AS total_amount
FROM `cineflix.customer` C
LEFT JOIN `cineflix.payment` P
  ON C.customer_id = P.customer_id
GROUP BY 1
ORDER BY 2 DESC
"""
df = client.query(query).to_dataframe()
df

,customer_id,total_amount
0,526,222.0
1,148,217.0
2,144,196.0
3,137,195.0
4,178,195.0
...,...,...
601,602,NaN
602,605,NaN
603,604,NaN
604,601,NaN


In [16]:
# Question    -- For each store, list the total number of films and the number of currently rented films.
query = """
SELECT
  s.store_id,
  COUNT(f.film_id) AS total_films,
  sum(
    CASE
      WHEN r.return_date IS NULL AND i.inventory_id IS NOT NULL THEN 1
      ELSE 0
      END) AS rented_films
FROM `cineflix.store` s
LEFT JOIN `cineflix.inventory` i
  ON s.store_id = i.store_id
LEFT JOIN `cineflix.rental` r
  ON r.inventory_id = i.inventory_id
LEFT JOIN `cineflix.film` f
  ON f.film_id = i.film_id
GROUP BY 1
ORDER BY 2
"""
df = client.query(query).to_dataframe()
df

,store_id,total_films,rented_films
0,3,0,0
1,4,0,0
2,1,7923,92
3,2,8122,92


# **Cross Join**